In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

### Libraries

In [ ]:
from math import sqrt

import torch
import seaborn as sns
from qgsw.logging import getLogger, setup_root_logger
from qgsw.specs import defaults
import gc

torch.backends.cudnn.deterministic = True
torch.set_grad_enabled(False)
sns.set_theme("notebook",style="dark")
gratio = (1+sqrt(5))/2

specs = defaults.get()

setup_root_logger(1)
logger = getLogger(__name__)

### Parameters

In [ ]:
from qgsw.configs.core import Configuration
from qgsw.forcing.wind import WindForcing


config = Configuration.from_toml("../output/g5k/param_optim/_config.toml")

H = config.model.h
g_prime = config.model.g_prime
H1, H2 = H[0], H[1]
g1, g2 = g_prime[0], g_prime[1]
beta_plane = config.physics.beta_plane
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

wind = WindForcing.from_config(
    config.windstress,
    config.space,
    config.physics,
)
tx, ty = wind.compute()
p = 4

In [ ]:
H1_,H2_ = 150, 1350 #600, 900 # H[0],H[1]
H_ = torch.tensor([H1_,H2_],**specs)
g1_, g2_ = g_prime[0],g_prime[1]*1
g_prime_ = torch.tensor([g1_,g2_],**specs)

### Space

In [ ]:
from qgsw.spatial.core.discretization import SpaceDiscretization3D


space = SpaceDiscretization3D.from_config(
    config.space,
    config.model,
)
dx, dy = space.dx, space.dy

In [ ]:
from qgsw.solver.boundary_conditions.base import Boundaries


def get_psi_slices(imin:int,imax:int,jmin:int,jmax:int) -> tuple[slice]:
    return  [slice(imin, imax + 1), slice(jmin, jmax + 1)]

def extract_psi_w_(psi: torch.Tensor,imin:int,imax:int,jmin:int,jmax:int) -> torch.Tensor:
    """Extract psi."""
    psi_slices_w = get_psi_slices(imin-p,imax+p,jmin-p,jmax+p)
    return psi[..., *psi_slices_w]


def extract_psi_bc(psi: torch.Tensor) -> Boundaries:
    """Extract psi."""
    return Boundaries.extract(psi, p, -p - 1, p, -p - 1, 2)

### Simulation

In [ ]:
dt = 7200
n_steps_per_cyle = 250
separation = 250
comparison_interval = 1
n_cycles = 12

### Outputs

In [ ]:
save_videos = False

### RMSE

In [ ]:
from qgsw.solver.finite_diff import grad_perp, laplacian
from qgsw.utils.reshaping import crop


def rmse(f: torch.Tensor, f_ref: torch.Tensor) -> torch.Tensor:
    """RMSE."""
    return (f - f_ref).square().mean().sqrt() / f_ref.square().mean().sqrt()

def grad_rmse(f:torch.Tensor, f_ref:torch.Tensor) -> torch.Tensor:
    """Gradient RMSE."""
    u,v = grad_perp(f)
    u/=dy
    v/=dx
    u_ref,v_ref = grad_perp(f_ref)
    u_ref/=dy
    v_ref/=dx

    return ((u-u_ref).square().mean()+(v-v_ref).square().mean()).sqrt() / (u_ref.square().mean()+v_ref.square().mean()).sqrt()

def vorticity_rmse(f:torch.Tensor, f_ref:torch.Tensor) -> torch.Tensor:
    """Vorticity RMSE."""
    omega = crop(laplacian(f,dx,dy),1)
    omega_ref = crop(laplacian(f_ref,dx,dy),1)

    return (omega - omega_ref).square().mean().sqrt() / omega_ref.square().mean().sqrt()

## Initial condition

In [ ]:
from qgsw.fields.variables.tuples import UVH
from qgsw.masks import Masks
from qgsw.models.qg.stretching_matrix import compute_A
from qgsw.models.qg.uvh.projectors.core import QGProjector
from qgsw.utils import covphys


P = QGProjector(
    A=compute_A(H=H, g_prime=g_prime),
    H=H.unsqueeze(-1).unsqueeze(-1),
    space=space,
    f0=beta_plane.f0,
    masks=Masks.empty(
        nx=config.space.nx,
        ny=config.space.ny,
    ),
)
uvh0 = UVH.from_file("../output/g5k/param_optim/_data_startup.pt")
psi_start = P.compute_p(covphys.to_cov(uvh0, dx, dy))[0] / beta_plane.f0

### Full-domain model

In [ ]:
from qgsw.models.qg.psiq.core import QGPSIQ


model_3l = QGPSIQ(
    space_2d=space.remove_h(),
    H=H,
    beta_plane=config.physics.beta_plane,
    g_prime=g_prime,
)
model_3l.set_wind_forcing(tx, ty)
model_3l.masks = Masks.empty_tensor(
    model_3l.space.nx,
    model_3l.space.ny,
    device=specs["device"],
)
model_3l.bottom_drag_coef = bottom_drag_coef
model_3l.slip_coef = slip_coef
model_3l.dt = dt
y0 = model_3l.y0

## OBC models

### Base

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import TypeVar

from matplotlib import pyplot as plt
import numpy as np

from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.spatial.core.discretization import SpaceDiscretization2D

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T]):
    results_paths = Path("../output/g5k/param_optim")

    instantiated=False

    losses:dict[str,list[list[torch.Tensor]]] = {}
    
    ijs:tuple[int,int,int,int] = None
    save_states = False
    show=True

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.losses = {
            "rmse": [],"grad_rmse": [], "vorticity_rmse": [],
        }
        self.states:dict[str,list[list[torch.Tensor]]] = {
            "psi1": []
        }

    def _set_params(self) -> None:
        space = self.model.space
        self.model.y0 = y0
        self.model.masks = Masks.empty_tensor(
            space.nx,
            space.ny,
            device=specs["device"],
        )
        self.model.bottom_drag_coef = 0
        self.model.wide = True
        self.model.slip_coef = slip_coef
        self.model.dt = dt
        
    def load(self, imin:int,imax:int,jmin:int,jmax:int)-> dict:
        indices = f"_{imin}_{imax}_{jmin}_{jmax}.pt"
        file = self.results_paths.joinpath(self.prefix+indices)
        if isinstance(r:=torch.load(file),dict):
            return r["results"]
        return r
    def new_cycle(self) -> None:
        super().new_cycle()
        if self.save_states:
            for s in self.states.values():
                s.append([])

        for loss in self.losses.values():
            loss.append([])
            
    def add_loss(self, loss_value:float,loss_name:str) -> None:
        self.losses[loss_name][-1].append(loss_value)
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        if not self.show:
            return
        if ax is None:
            ax = plt.gca()
        cycles = [cycle] if cycle is not None else list(range(len(self.losses[loss_name])))
        time_offset= 0
        for i,c in enumerate(cycles):
            times = self.model.dt*np.arange(len(self.losses[loss_name][c]))/3600/24 + time_offset
            time_offset = times[-1] + self.model.dt/3600/24
            loss =  np.array(self.losses[loss_name][c])
            kwargs = self.plot_kwargs.copy()
            if i!= 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")
    def step(self) -> None:
        super().step()
        if self.save_states:
            self.states["psi1"].append(self.model.psi[:,:1])
            
        
M = TypeVar("M", bound=ModelWrapperOBC[QGPSIQCore])

class ModelsManagerOBC(ModelsManager[M]):

    loss_fn: dict[str, Callable[[torch.Tensor,torch.Tensor], torch.Tensor]]= {
        "rmse":rmse,
        "grad_rmse":grad_rmse,
        "vorticity_rmse":vorticity_rmse
    }

    losses = list(loss_fn.keys())

    def __init__(self, *mw:M) -> None:
        super().__init__(*mw)
        for mw in self.model_wrappers:
            mw.instantiated=True
        self.ijs = self.model_wrappers[0].ijs


    @property
    def ijs(self) -> tuple[int,int,int,int]:
        return self.model_wrappers[0].ijs
    @ijs.setter
    def ijs(self,ijs:tuple[int,int,int,int]) -> None:
        self.loop_over_models(lambda mw: setattr(mw,"ijs",ijs))

    def compute_loss(self, psi_ref:torch.Tensor) -> None:
        for loss_name in self.losses:
            self.loop_over_models(
                lambda mw: mw.add_loss(self.loss_fn[loss_name](mw.model.psi[0,0],psi_ref[0,0]).cpu().item(),loss_name)
            )
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        self.loop_over_models(lambda mw: mw.plot_loss(loss_name=loss_name,ax=ax,cycle=cycle))

### Reduced gravity

In [ ]:
from torch import Tensor
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate
from qgsw.utils.interpolation import QuadraticInterpolation
from qgsw.utils.reshaping import crop


class ReducedGravity(ModelWrapperOBC[QGPSIQ]):
    prefix = None
    color = "black"
    label="Reduced Gravity"
    H=H[:1]
    g_prime = g_prime[:1]*g_prime[1:2]/(g_prime[:1]+g_prime[1:2])
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQ(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self._set_params()
        # self.model.wind_scaling = H[:1].item()
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1/g1+1/H1/g2)*psi[...,1:-1,1:-1]) + beta_effect
    
    def setup(self, psis: list[torch.Tensor],times:list[torch.Tensor],beta_effect_w:torch.Tensor) -> None:
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
            Boundaries.extract(
                self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
            )
            for psi in psis
        ]
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],beta_effect_w),p-1))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
class ReducedGravityPert(ReducedGravity):
    H=H_[:1]
    g_prime_=g_prime_[:1]*g_prime_[1:2]/(g_prime_[:1]+g_prime_[1:2])
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1_/g1_+1/H1_/g2_)*psi[...,1:-1,1:-1]) + beta_effect

### SurfML

In [ ]:
from torch._tensor import Tensor
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.decomposition.exp_exp.core import GaussianExpBasis
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.utils.tensor_dict import change_specs



class SurfML(ModelWrapperOBC[QGPSIQRGPsi2TransportDR]):
    prefix = "results_mixed_rg_ro_ge_nopert"
    color="orange"
    label="GaussBarotropic - RG - NoPert - DR"
    save_video = False
    H = H[:2]
    g_prime=g_prime[:2]
    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.states["psi2"] = []
        
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQRGPsi2TransportDR(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self._set_params()
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load(*self.ijs)

        imin,imax,jmin,jmax = self.ijs

        space_slice = space.remove_h().slice(
            imin,imax+1,jmin,jmax+1
        )
        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        self.A = compute_A_tilde(self.H,self.g_prime,alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        self.basis: GaussianExpBasis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            self.basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        self.basis.set_coefs(coefs)
        self._fpsi2 = self.basis.localize(
            space_slice.psi.xy.x,space_slice.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),p-1))
        self.model.alpha = alpha
        self.model.basis = self.basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
            self.states["psi2"] = [self._fpsi2(self.model.time)[None,None,...]]

    def step(self) -> None:
        super().step()
        if self.save_states:
            self.states["psi2"].append(self._fpsi2(self.model.time)[None,None,...])
class SurfMLPert(SurfML):
    H=H_[:2]
    g_prime=g_prime_[:2]

### Forced

In [ ]:
from qgsw.decomposition.wavelets import WaveletBasis
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced
from qgsw.spatial.core.discretization import SpaceDiscretization2D

Heq = H[:1]*H[1:2]/(H[:1]+H[1:2])
Heq_ = H_[:1]*H_[1:2]/(H_[:1]+H_[1:2])
class VarDyn(ModelWrapperOBC[QGPSIQForced]):
    prefix = "results_forced_rg_dr"
    color="brown"
    label="Forced DR"
    save_video = False
    H = Heq
    g_prime = g_prime[1:2]
    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.states["forcing"] = []
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForced(
            space_2d=space_2d,
            H=self.H,
            beta_plane=beta_plane,
            g_prime=self.g_prime,
        )
        self.model.wind_scaling = H[:1].item()
        self._set_params()
        self.coefs = {}
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq/g2)*psi[...,1:-1,1:-1]
        ) + beta_effect

    def setup(self, psis: list[Tensor], times: list[Tensor], beta_effect_w: Tensor) -> None:
        res = self.load(*self.ijs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        self.basis:WaveletBasis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.basis.set_coefs(coefs)
        try:
            self.basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cyle],**specs))
        except:... 
        if self.save_params:
            self.coefs[self.cycle] = coefs
            
        self.wv = self.basis.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )

        psi0 = psis[0]
        psi_bcs = [extract_psi_bc(psi[:,:1]) for psi in psis]
        q_bcs = [
                Boundaries.extract(
                    self.compute_q(psi[:, :1],beta_effect_w), 2, -3, 2, -3, 3
                )
                for psi in psis
            ]

        self.model.set_psiq(crop(psi0[:,:1],p), crop(self.compute_q(psi0[:,:1],beta_effect_w),p-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        
        if self.save_states:
            self.states["psi1"] = [self.model.psi[:,:1]]
            self.states["forcing"] = [crop(self.wv(self.model.time)[None,None,...],p)]
        
    def step(self) -> None:
        self.model.forcing = self.wv(self.model.time)
        super().step()
        if self.save_states:
            self.states["forcing"].append(crop(self.wv(self.model.time)[None,None,...],p))


class VarDynPert(VarDyn):
    H = Heq_
    g_prime = g_prime_[1:2]

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.model.wind_scaling = H_[:1].item()

    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq_/g2_)*psi[...,1:-1,1:-1]
        ) + beta_effect


In [ ]:
from qgsw.logging.utils import box, step
from qgsw.solver.finite_diff import grad


def main_forecast(imin:int, imax:int,jmin:int,jmax:int, *wrappers:ModelWrapperOBC[QGPSIQ]) -> tuple[ModelsManagerOBC,list[torch.Tensor]]:
    extract_psi_w = lambda psi: extract_psi_w_(psi,imin,imax,jmin,jmax)

    model_3l.reset_time()
    model_3l.set_psi(psi_start)

    space_slice_w = SpaceDiscretization2D.from_coords(
        x_1d=P.space.remove_h().omega.xy.x[imin - p + 1 : imax + p, 0],
        y_1d=P.space.remove_h().omega.xy.y[0, jmin - p + 1 : jmax + p],
    )
    y_w = space_slice_w.q.xy.y[0, :].unsqueeze(0)
    beta_effect_w = beta_plane.beta * (y_w - y0)

    models = ModelsManagerOBC(
        *wrappers
    )
    models.ijs = (imin,imax,jmin,jmax)

    models.set_wind_forcing(tx[imin:imax, jmin : jmax + 1],ty[imin : imax + 1, jmin:jmax])

    psi0s = {}
    gc.collect()


    for c in range(n_cycles):
        torch.cuda.reset_peak_memory_stats()
        models.new_cycle()
        model_3l.reset_time()

        times = [model_3l.time.item()]

        psi0 = extract_psi_w(model_3l.psi[:,:2])

        psi0s[c] = psi0

        psis = [psi0]

        for _ in range(1, n_steps_per_cyle):
            model_3l.step()

            times.append(model_3l.time.item())

            psi = extract_psi_w(model_3l.psi[:,:2])

            psis.append(psi)

        psi0_ = model_3l.psi.clone()
        q0_ = model_3l.q.clone()

        for _ in range(1, n_steps_per_cyle):
            model_3l.step()

            times.append(model_3l.time.item())

            psi = extract_psi_w(model_3l.psi[:,:2])

            psis.append(psi)
            

        models.reset_time()

        models.setup(psis,times,beta_effect_w)

        models.compute_loss(crop(psis[0],p))


        for n in range(1,n_steps_per_cyle):
            models.step()
            models.compute_loss(crop(psis[n],p))


        for n in range(1,n_steps_per_cyle):
            models.step()
            models.compute_loss(crop(psis[n+n_steps_per_cyle-1],p))

        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.empty_cache()

        max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
        msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
        logger.info(box(msg_mem, style="round"))


        model_3l.set_psiq(psi0_,q0_)

        for _ in range(separation):
            model_3l.step()

    return models, psis

In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

def show_results(imin:int,imax:int,jmin:int,jmax:int,models:ModelsManagerOBC)-> None:
    show_grad = True
    show_vort = True

    fig,axs = plots.subplots(1+show_grad+show_vort,1,figsize=(21,5+5*show_grad+5*show_vort))
    fig.suptitle(f"QG3L - [{imin}, {imax}] x [{jmin}, {jmax}]")
    plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] + show_vort*[ "Vorticity RMSE"],axs=axs)
    models.plot_loss(loss_name="rmse",ax=axs[0,0])
    plots.clamp_ylims(0,1,axs[0,0])
    axs[0,0].legend(loc="upper left",prop={'size': 8})
    if show_grad:
        models.plot_loss(loss_name="grad_rmse",ax=axs[1,0])
        plots.clamp_ylims(0,1,axs[1,0])
        axs[1,0].legend(loc="upper left",prop={'size': 8})
    if show_vort:
        models.plot_loss(loss_name="vorticity_rmse",ax=axs[-1,0])
        plots.clamp_ylims(0,1,axs[-1,0])
        axs[-1,0].legend(loc="upper left",prop={'size': 8})

# Forecast

## [32, 96] x [256, 384]

In [ ]:
imin, imax = 32, 96
jmin, jmax = 256, 384


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravity(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDyn(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_gamma1000000000_obstrack_s250"

surfml_g100 = SurfML(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_gamma100_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Projected ψ₂"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z1, psis = main_forecast(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_g100,psi2)

In [ ]:
from copy import deepcopy


losses_z1 = {
    "rg": deepcopy(rg.losses),
    "vardyn": deepcopy(forced_g1000000000.losses),
    "surfml": deepcopy(surfml_g100.losses),
    "psi2": deepcopy(psi2.losses),
}

In [ ]:
show_results(imin,imax,jmin,jmax,models_z1)

In [ ]:
losses_z1 = torch.load("../output/tmp.pt")

## [112, 176] x [64, 192]

In [ ]:
imin, imax = 112, 176
jmin, jmax = 64, 192


space_slice = SpaceDiscretization2D.from_coords(
    x_1d=P.space.remove_h().omega.xy.x[imin : imax + 1, 0],
    y_1d=P.space.remove_h().omega.xy.y[0, jmin : jmax + 1],
)

rg = ReducedGravity(space_slice)
rg.linestyle="solid"

forced_g1000000000 = VarDyn(space_slice)
forced_g1000000000.label = "Forced - 1000000000"
forced_g1000000000.linestyle = "solid"
forced_g1000000000.prefix = "results_forced_rg_dr_gamma1000000000_obstrack_s250"

surfml_g100 = SurfML(space_slice)
surfml_g100.label = "SurfML -  100"
surfml_g100.color = "navy"
surfml_g100.linestyle = "solid"
surfml_g100.prefix = "results_surfml_gamma100_obstrack_s250"

psi2 = SurfML(space_slice)
psi2.label = "Projected ψ₂"
psi2.color="green"
psi2.linestyle="solid"
psi2.prefix = "results_psi2_o1000_s250"
psi2.results_paths = Path("../output/local/param_optim")

models_z2, psis = main_forecast(imin,imax,jmin,jmax, rg, forced_g1000000000, surfml_g100,psi2)

In [ ]:
from copy import deepcopy


losses_z2 = {
    "rg": deepcopy(rg.losses),
    "vardyn": deepcopy(forced_g1000000000.losses),
    "surfml": deepcopy(surfml_g100.losses),
    "psi2": deepcopy(psi2.losses),
}

In [ ]:
show_results(imin,imax,jmin,jmax,models_z2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker

forced_g1000000000.label = "RG-F"
surfml_g100.label = "RG-SI"
psi2.label = "Projected ψ₂"
rg.label = "RG0"

fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left  = Size.Fixed(0.8)
ax_width  = Size.Fixed(5.0)
gap       = Size.Fixed(0.1)
pad_right = Size.Fixed(0.3)

row_height = Size.Fixed(5 / gratio)
row_gap    = Size.Fixed(0.4)   # vertical gap between the two rows

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [
    Size.Fixed(0.6),   # bottom margin
    row_height,        # bottom row (row 1)
    row_gap,           # gap between rows
    row_height,        # top row (row 2)
    Size.Fixed(0.4),   # top margin
]

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

# Bottom row  (ny=1)
ax1_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                       sharey=ax1_bot)

# Top row  (ny=3)
ax1_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=3),
                        sharex=ax1_bot)
ax2_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=3),
                       sharey=ax1_top, sharex=ax2_bot)

times = np.arange(2*n_steps_per_cyle-1) * dt

# ── plotting helper ───────────────────────────────────────────────────────────
# def plot_on(ax_summer, ax_winter, losses_dict):

for mw, loss_name in zip([rg, forced_g1000000000, surfml_g100,psi2], ['rg', "vardyn", "surfml","psi2"]):
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[loss_name]["rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    t = times / 3600 / 24
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax1_top.plot(t, m, **mw.plot_kwargs)
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[loss_name]["rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax2_top.plot(t, m, **mw.plot_kwargs)

for mw, loss_name in zip([rg, forced_g1000000000, surfml_g100,psi2], ['rg', "vardyn", "surfml","psi2"]):
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[loss_name]["grad_rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    t = times / 3600 / 24
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax1_bot.plot(t, m, **mw.plot_kwargs)
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[loss_name]["grad_rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax2_bot.plot(t, m, **mw.plot_kwargs)

# ── shared spine / tick style ─────────────────────────────────────────────────
def style_ax(ax, hide_yticks=False, hide_xticks=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_bounds(0, 40)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("outward", 5))
    ax.spines["left"].set_position(("outward", 5))
    ax.tick_params(axis='both', labelsize="large")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
    if hide_yticks:
        ax.tick_params(labelleft=False)
    if hide_xticks:
        ax.tick_params(labelbottom=False)
        ax.spines["bottom"].set_visible(False)

style_ax(ax1_bot)
style_ax(ax2_bot, hide_yticks=True)
style_ax(ax1_top, hide_xticks=True)          # no x-axis on top row
style_ax(ax2_top, hide_yticks=True, hide_xticks=True)

# ── labels & titles ───────────────────────────────────────────────────────────
ax1_top.set_title("Z1", fontsize="x-large")
ax2_top.set_title("Z2",  fontsize="x-large")

ax1_bot.set_xlabel("Time [days]", fontsize="large")
ax2_bot.set_xlabel("Time [days]", fontsize="large")

ax1_top.set_ylabel("Stream function nRMSE", fontsize="large")
ax1_bot.set_ylabel("Velocity nRMSE", fontsize="large")

ax1_top.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax1_bot.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax2_top.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax2_bot.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)

ax1_top.legend(fontsize="large")
plots.clamp_ylims(0, 1, ax1_top)
plots.clamp_ylims(0, 1, ax1_bot)
plots.set_ylims(0, None, ax1_bot)
plots.set_ylims(0, None, ax1_top)

# fig.savefig("../output/images/qg3l_all_rmses_forecast.svg", bbox_inches="tight")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker

forced_g1000000000.label = "RG-F"
surfml_g100.label = "RG-SI"
psi2.label = "Projected ψ₂"
rg.label = "RG0"

fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left  = Size.Fixed(0.8)
ax_width  = Size.Fixed(5.0)
gap       = Size.Fixed(0.1)
pad_right = Size.Fixed(0.3)

row_height = Size.Fixed(5 / gratio)
row_gap    = Size.Fixed(0.4)   # vertical gap between the two rows

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [
    Size.Fixed(0.6),   # bottom margin
    row_height,        # bottom row (row 1)
    row_gap,           # gap between rows
    row_height,        # top row (row 2)
    Size.Fixed(0.4),   # top margin
]

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

# Bottom row  (ny=1)
ax1_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                       sharey=ax1_bot)

# Top row  (ny=3)
ax1_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=3),
                        sharex=ax1_bot)
ax2_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=3),
                       sharey=ax1_top, sharex=ax2_bot)

times = np.arange(2*n_steps_per_cyle-1) * dt

# ── plotting helper ───────────────────────────────────────────────────────────
# def plot_on(ax_summer, ax_winter, losses_dict):

for mw, loss_name in zip([rg, forced_g1000000000, surfml_g100,psi2], ['rg', "vardyn", "surfml","psi2"]):
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[loss_name]["rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    t = times / 3600 / 24
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax1_top.plot(t, m, **mw.plot_kwargs)
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[loss_name]["rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax2_top.plot(t, m, **mw.plot_kwargs)

for mw, loss_name in zip([rg, forced_g1000000000, surfml_g100,psi2], ['rg', "vardyn", "surfml","psi2"]):
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[loss_name]["grad_rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    t = times / 3600 / 24
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax1_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax1_bot.plot(t, m, **mw.plot_kwargs)
    ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[loss_name]["grad_rmse"]])
    ms   = torch.mean(ls[~torch.isnan(ls).any(dim=1)], dim=0)
    stds = torch.std(ls[~torch.isnan(ls).any(dim=1)],  dim=0)
    m, s = ms.cpu().numpy(), stds.cpu().numpy()
    ax2_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
    ax2_bot.plot(t, m, **mw.plot_kwargs)

# ── shared spine / tick style ─────────────────────────────────────────────────
def style_ax(ax, hide_yticks=False, hide_xticks=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_bounds(0, 40)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("outward", 5))
    ax.spines["left"].set_position(("outward", 5))
    ax.tick_params(axis='both', labelsize="large")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
    if hide_yticks:
        ax.tick_params(labelleft=False)
    if hide_xticks:
        ax.tick_params(labelbottom=False)
        ax.spines["bottom"].set_visible(False)

style_ax(ax1_bot)
style_ax(ax2_bot, hide_yticks=True)
style_ax(ax1_top, hide_xticks=True)          # no x-axis on top row
style_ax(ax2_top, hide_yticks=True, hide_xticks=True)

# ── labels & titles ───────────────────────────────────────────────────────────
ax1_top.set_title("Z1", fontsize="x-large")
ax2_top.set_title("Z2",  fontsize="x-large")

ax1_bot.set_xlabel("Time [days]", fontsize="large")
ax2_bot.set_xlabel("Time [days]", fontsize="large")

ax1_top.set_ylabel("Stream function nRMSE", fontsize="large")
ax1_bot.set_ylabel("Velocity nRMSE", fontsize="large")

ax1_top.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax1_bot.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax2_top.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)
ax2_bot.vlines([250*7200/3600/24],0,1,linestyle="dashdot",color="grey",alpha=0.5)

ax1_top.legend(fontsize="large")
plots.clamp_ylims(0, 1, ax1_top)
plots.clamp_ylims(0, 1, ax1_bot)
plots.set_ylims(0, None, ax1_bot)
plots.set_ylims(0, None, ax1_top)

# fig.savefig("../output/images/qg3l_all_rmses_forecast.svg", bbox_inches="tight")

In [ ]:
print("Z1 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,:250]) for k,v in losses_z1.items()
})
print("Z2 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,:250]) for k,v in losses_z2.items()
})

In [ ]:
print("Z1 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,:250]) for k,v in losses_z1.items()
})
print("Z2 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,:250]) for k,v in losses_z2.items()
})

In [ ]:
print("Z1 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,250:]) for k,v in losses_z1.items()
})
print("Z2 RMSEs: ", {
    k:np.mean(np.array(v["rmse"])[~np.isnan(np.array(v["rmse"])).any(axis=1)][:,250:]) for k,v in losses_z2.items()
})

In [ ]:
print("Z1 UV RMSEs: ", {
    k:np.mean(np.array(v["grad_rmse"])[~np.isnan(np.array(v["grad_rmse"])).any(axis=1)]) for k,v in losses_z1.items()
})
print("Z2 UV RMSEs: ", {
    k:np.mean(np.array(v["grad_rmse"])[~np.isnan(np.array(v["grad_rmse"])).any(axis=1)]) for k,v in losses_z2.items()
})